In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

#Load the dataset
data = load_breast_cancer()

x = pd.DataFrame(
    data.data,
    columns=data.feature_names
)

y = pd.Series(
    (data.target == 0).astype(int),
    name="malignant"
)

print(y.value_counts())

print("Feature matrix slope",x.shape)
print("Target Shape:", y.shape)
print("Class Names:",data.target_names)

In [ ]:
# Examine the distribution

class_counts = y.value_counts().sort_index()
class_distribution = pd.DataFrame({
    "Class":data.target_names,
    "Count":class_counts.values,
    "Probability":class_counts.values/len(y)
})

print(class_distribution)


In [ ]:
class_distribution.plot(
    x="Class",
    y="Count",
    kind="bar",
    legend=False,
    color=["tomato", "steelblue"]
)

plt.ylabel("Number of observations")
plt.title("Class Distribution")
plt.xticks(rotation=0)
plt.show()

In [ ]:
X_train, X_test,y_train,y_test = train_test_split(
    x,y,test_size=0.2,random_state=42,stratify=y
)

print("Training size:",len(y_train))
print("Testing size:",len(y_test))

print("\nTraining proportions:")
print(y_train.value_counts(normalize=True).sort_index())

print("\nTesting proportions:")
print(y_test.value_counts(normalize=True).sort_index())


In [ ]:
#Train logistic regression
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)

model.fit(X_train,y_train)

In [ ]:
probabilities = model.predict_proba(X_test)
# For each x_test we get a probability
print(probabilities[:5])

In [ ]:
print(probabilities[:5].sum(axis=1))

In [ ]:
results = pd.DataFrame({
    "Actual_class":y_test.values,
    "P_malignant":probabilities[:,0],
    "P_benign":probabilities[:,1]
})

print(results.head(10))

In [ ]:
threshold = 0.5

results["Predicted_malignant"] = (
    results["P_malignant"] >=  threshold
).astype(int)

results["Predicted_label"] = results["Predicted_malignant"].map({
    1:"Malignant",
    0:"Benign"
})

print(
    results[
        [
        "Actual_class",
        "P_malignant",
        "Predicted_label"]
    ].head(10)
)

In [ ]:
for threshold in [0.3,0.5,0.7]:
    predictions = (
        results["P_malignant"] >=threshold
    ).astype(int)

    print(
        f"Threshold = {threshold}:"
        f"Predicted Malignant cases = {predictions.sum()}"
    )

In [ ]:
# Connstruct the confusion matrix
actual_malignant = (y_test.values==0).astype(int)

for threshold in [0.3,0.5,0.7]:
    predicted_malignant = (
        probabilities[:,0] >= threshold
    ).astype(int)

    cm = confusion_matrix(
        actual_malignant,
        predicted_malignant
    )

    print(f"\nThreshold = {threshold}")
    print(cm)

In [ ]:
# y_pred = probabilities[:1]

In [ ]:
# Accuracy, Precision, Recall, F1 Score, Specificity

# from sklearn.metrics import (
#     accuracy_score,
#     precision_score,
#     recall_score,
#     f1_score
# )

# print("Accuracy:", accuracy_score(y_test,y_pred))
# print("Precision:",precision_score(y_test,y_pred))
# print("Recall/Sensitivity:",recall_score(y_test,y_pred))
# print("F1 Score",f1_score(y_test,y_pred))

In [ ]:
metric_results =[]

for threshold in [0.1,0.3,0.5,0.7,0.9]:
    y_pred_threshold  =(
        probabilities[:,1] >= threshold
    ).astype(int)

    tn,fp,fn,tp = confusion_matrix(
        y_test,
        y_pred_threshold,
        labels=[0,1]
    ).ravel()

    accuracy = (tp+tn) / (tp+tn + fp+fn)
    precision = tp / (tp+fp) if (tp+fp) > 0 else 0 
    recall = tp / (tp+fn) if (tp+fn) > 0 else 0
    specificity = tn/(tn+fp) if (tn+fp) > 0 else 0 

    f1_score =(
        2*(precision*recall) / (precision+recall) if (precision + recall)>0 else 0 
    )

    print

    metric_results.append({
        "Threshold":threshold,
        "TN":tn,
        "FP":fp,
        "TP":tp,
        "FN":fn,
        "Accracy":accuracy,
        "Precision":precision,
        "Recall":recall,
        "Specificity":specificity,
        "F1 Score":f1_score

    })
metric_results_df = pd.DataFrame(metric_results)
display(metric_results_df.round(3))

,Threshold,TN,FP,TP,FN,Accracy,Precision,Recall,Specificity,F1 Score
0,0.1,67,5,41,1,0.947,0.891,0.976,0.931,0.932
1,0.3,71,1,41,1,0.982,0.976,0.976,0.986,0.976
2,0.5,71,1,39,3,0.965,0.975,0.929,0.986,0.951
3,0.7,72,0,38,4,0.965,1.000,0.905,1.000,0.950
4,0.9,72,0,33,9,0.921,1.000,0.786,1.000,0.880


In [ ]:
#Make ROC/AUC Dataset
plt. plot(
    threshold,
    recall_values[:-1],
    label="Recall/Sensitivity"
)